In [1]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv()  # 加载.env文件里的变量
# print(os.getenv("DEEPSEEK_API_KEY"))  # 现在可以正常读取了

llm = ChatOpenAI(
        model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
        temperature=0,
    )

In [2]:
import numpy as np
import pandas as pd
import json
import io
import inspect
import requests
from langchain_core.tools import tool

@tool
def get_weather(loc):
    """
    查询即时天气函数
    :param loc: 必要参数，字符串类型，用于表示查询天气的具体城市名称
    注意， 中国的城市需要用对应城市的英文名称代替，
    :return : OpenWeather API 查询即时天气的结果，具体URL请求地址为: https://api.openweathermap.org/data/2.5/weather
    返回结果对象类型为解析之后的JSON格式对象，并用字符串形式进行表示，其中包含了全部重要的天气信息
    """
    
    url=os.getenv("WEATHER_API_URL")
    
    params={
        "q":loc,
        'appid':os.getenv('WEATHER_API_KEY'),
        'units':'metric',
        'lang':'zh_cn'
    }
    
    response=requests.get(url,params=params)
    
    data=response.json()
    return json.dumps(data)

In [3]:
from langgraph.prebuilt import ToolNode

tools=[get_weather]
toolNode=ToolNode(tools)

pip install langchainhub


In [4]:
from langchain.agents import create_agent

agent = create_agent(llm, tools, system_prompt="你是一个善于调用工具来回答用户问题的助手")


In [5]:
result=agent.invoke({
    'messages':["大连今天的天气"]
})
result['messages'][-1].content

'我已经查询到大连今天的天气情况，以下是详细信息：\n\n### 🌤️ 大连今日天气\n\n- **天气状况**：多云\n- **气温**：27.0°C（体感温度 28.7°C）\n- **湿度**：69%\n- **气压**：1011 hPa\n- **风速**：3 m/s（北风，风向 350°）\n- **云量**：35%\n- **能见度**：10 公里\n\n### 📌 温馨提示\n- 今天天气多云，气温较为舒适，体感略热，适合外出活动。\n- 湿度适中，风力不大，整体天气状况良好。\n- 建议外出时注意防晒，适当补水。\n\n祝你有愉快的一天！☀️'

In [6]:
for chunk in agent.stream({
    'messages':["大连今天的天气"]
},stream_mode="values"):
    chunk['messages'][-1].pretty_print()

================================ Human Message =================================

大连今天的天气
================================== Ai Message ==================================

我来帮您查询大连今天的天气情况。
Tool Calls:
  get_weather (call_00_g9OHUMXydUnCflQfGIdQ8578)
 Call ID: call_00_g9OHUMXydUnCflQfGIdQ8578
  Args:
    loc: Dalian
================================= Tool Message =================================
Name: get_weather

{"coord": {"lon": 121.6022, "lat": 38.9122}, "weather": [{"id": 802, "main": "Clouds", "description": "\u591a\u4e91", "icon": "03d"}], "base": "stations", "main": {"temp": 26.96, "feels_like": 28.71, "temp_min": 26.96, "temp_max": 26.96, "pressure": 1011, "humidity": 69, "sea_level": 1011, "grnd_level": 1006}, "visibility": 10000, "wind": {"speed": 3, "deg": 350}, "clouds": {"all": 35}, "dt": 1788145407, "sys": {"type": 1, "id": 9679, "country": "CN", "sunrise": 1788124876, "sunset": 1788172041}, "timezone": 28800, "id": 1814087, "name": "Dalian", "cod": 200}
=================

In [8]:
# result=agent.invoke({
#     'messages':["查一下今天大理，昆明和丽江哪个城市的气温最低"]
# })
# result['messages'][-1].content


for chunk in agent.stream({
    'messages':["查一下今天大理，昆明和丽江哪个城市的气温最低"]
},stream_mode="values"):
    chunk['messages'][-1].pretty_print()

================================ Human Message =================================

查一下今天大理，昆明和丽江哪个城市的气温最低
================================== Ai Message ==================================

好的，我来帮您查询大理、昆明和丽江这三个城市的天气情况。让我同时查询这三个城市的气温数据。
Tool Calls:
  get_weather (call_00_UI9KJiuoqenZ8JkKgVQg9360)
 Call ID: call_00_UI9KJiuoqenZ8JkKgVQg9360
  Args:
    loc: Dali
  get_weather (call_01_6CRRipfZqLXPHgEtdJ4V0082)
 Call ID: call_01_6CRRipfZqLXPHgEtdJ4V0082
  Args:
    loc: Kunming
  get_weather (call_02_W63H1WyRwdvf6HJXQyaw6588)
 Call ID: call_02_W63H1WyRwdvf6HJXQyaw6588
  Args:
    loc: Lijiang
================================= Tool Message =================================
Name: get_weather

{"coord": {"lon": 100.2207, "lat": 26.8688}, "weather": [{"id": 804, "main": "Clouds", "description": "\u9634\uff0c\u591a\u4e91", "icon": "04d"}], "base": "stations", "main": {"temp": 13.4, "feels_like": 13.35, "temp_min": 13.4, "temp_max": 13.4, "pressure": 1017, "humidity": 98, "sea_level": 1017, "grnd_l